# Campus Suite AI — repository-grounded 2B fine-tuning and export

This Colab trains a QLoRA adapter for **Campus Suite AI** from the supplied system prompt and real project workflows. It builds a small seed set from the inspected Campus Suite architecture, accepts additional JSON/JSONL data, evaluates behavior before and after training, and exports an adapter, an optional merged Hugging Face model, and a Q4_K_M GGUF when the selected architecture is supported by the checked-out llama.cpp converter.

**Run top to bottom in a GPU runtime.** Training, merging and GGUF export are separate stages; the adapter is the primary deliverable. A converter/runtime incompatibility stops export with diagnostics and preserves earlier outputs. Model metadata stays truthful: user-facing identity is Campus Suite AI, developed by Venkat and Shaan; a direct foundation-checkpoint question receives the actual checkpoint name.

Proposed starter: about 80 seed examples from actual routing, coding tools and agent workflow; target a reviewed 400–800 examples before a serious run. This notebook does not treat application logs as training data.


## 0. Runtime and controls

Select **Runtime → Change runtime type → GPU** before continuing. A T4 is suitable for QLoRA at a 2B scale with short sequences and micro-batch 1; 16–24 GB GPUs allow more headroom. Merge/export can need substantial RAM/disk space. This notebook checks CUDA, memory, Python/package versions, model config, tokenizer chat-template support, adapter targets and GGUF converter support before the relevant expensive step.

The model repository and packages are external dependencies. Review the selected model card/license and pin the printed model revision for reproducibility. Do not upload private student data, credentials, `.env` files or operational logs.


In [ ]:
# Keep Colab's CUDA-enabled PyTorch. Install the training stack without replacing torch.
import sys, subprocess, importlib.metadata as md, platform, json, os, re, gc, shutil
from pathlib import Path

print('Python:', platform.python_version())
try:
    import torch
except Exception as exc:
    raise RuntimeError('Colab PyTorch is unavailable. Select a Python GPU runtime and restart.') from exc
print('PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda, '| available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU required. Enable a Colab GPU runtime before installing/training.')
print('GPU:', torch.cuda.get_device_name(0), '| VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))

PACKAGES = [
    'transformers>=5.0.0,<6.0.0', 'peft>=0.18.0,<1.0.0', 'accelerate>=1.10.0,<2.0.0',
    'bitsandbytes>=0.47.0,<1.0.0', 'datasets>=4.0.0,<6.0.0', 'safetensors>=0.6.0',
    'sentencepiece>=0.2.0', 'packaging>=24.0', 'huggingface_hub>=0.34.0'
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade-strategy', 'only-if-needed', *PACKAGES], check=True)
print('Installed package versions:')
for pkg in ['transformers','peft','accelerate','bitsandbytes','datasets','safetensors','sentencepiece','huggingface-hub']:
    try: print(f'  {pkg}: {md.version(pkg)}')
    except md.PackageNotFoundError: raise RuntimeError(f'Required package missing after install: {pkg}')


In [ ]:
# Import/version and GPU feature preflight. No model weights downloaded yet.
import torch, transformers, peft, accelerate, bitsandbytes, datasets
from packaging.version import Version
from transformers import AutoConfig, AutoTokenizer
from peft import LoraConfig, get_peft_model

if Version(transformers.__version__) < Version('5.0.0'):
    raise RuntimeError(f'Transformers {transformers.__version__} is too old for the requested Qwen3.5 support.')
if Version(peft.__version__) < Version('0.18.0'):
    raise RuntimeError(f'PEFT {peft.__version__} is too old for the selected all-linear adapter path.')
if not torch.cuda.is_bf16_supported():
    print('GPU does not support BF16; training will use FP16.')
print('Preflight imports OK. Transformers', transformers.__version__, '| PEFT', peft.__version__)


## 1. Model and run configuration

The default is Qwen/Qwen3.5-2B, a roughly 2B open-weight instruction checkpoint consistent with the supplied notebook. It is Apache-2.0 according to its model page. The exact config, class, tokenizer, chat template and model revision are verified next; change `BASE_MODEL` only to a compatible causal language model and repeat the preflight. The notebook intentionally does not patch architecture metadata.


In [ ]:
BASE_MODEL = 'Qwen/Qwen3.5-2B'
MODEL_REVISION = 'main'  # Replace with a Hugging Face commit hash for reproducibility.
OUTPUT_ROOT = Path('/content/campus_suite_ai_output')
ADAPTER_DIR = OUTPUT_ROOT / 'campus-suite-ai-lora'
MERGED_DIR = OUTPUT_ROOT / 'campus-suite-ai-merged'
MAX_LENGTH = 2048
EPOCHS = 2
LEARNING_RATE = 1.0e-4
TRAIN_BATCH_SIZE = 1
EVAL_BATCH_SIZE = 1
GRAD_ACCUM = 8
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
SEED = 42
EVAL_FRACTION = 0.10
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

config = AutoConfig.from_pretrained(BASE_MODEL, revision=MODEL_REVISION, trust_remote_code=False)
archs = getattr(config, 'architectures', None) or []
model_type = getattr(config, 'model_type', '')
print('Model:', BASE_MODEL, '| revision:', MODEL_REVISION)
print('model_type:', model_type, '| architectures:', archs)
print('parameter config:', getattr(config, 'num_hidden_layers', '?'), 'layers,', getattr(config, 'hidden_size', '?'), 'hidden size')
if not archs or not any('ForCausalLM' in a for a in archs):
    raise RuntimeError(f'Expected a causal language model architecture; config reports {archs!r}.')
if 'qwen3_5' in model_type.lower() and not hasattr(transformers, 'Qwen3_5ForCausalLM'):
    raise RuntimeError('This Transformers build cannot import Qwen3_5ForCausalLM. Stop before downloading weights; update the Colab runtime/package compatibility.')

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, revision=MODEL_REVISION, use_fast=True, trust_remote_code=False)
if not tokenizer.chat_template:
    raise RuntimeError('Tokenizer has no chat template. Supply a compatible instruct checkpoint/template before training.')
if tokenizer.eos_token_id is None:
    raise RuntimeError('Tokenizer EOS token is missing; refusing to train with ambiguous turn boundaries.')
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
print('Tokenizer:', type(tokenizer).__name__, '| vocab:', len(tokenizer), '| EOS:', tokenizer.eos_token, '| template chars:', len(tokenizer.chat_template))


## 2. Upload the system prompt and optional JSON/JSONL data

Upload the supplied `Campus_VS_System_Prompt_FIXED.txt` and any additional `.json`, `.jsonl`, or `.ndjson` files when the picker opens. The fixed prompt is injected as the one system message for every sample, replacing any older system messages in uploaded data. JSON accepts one conversation or a list; JSONL accepts one conversation object per nonblank line. Each record uses `messages` with `user`, `assistant`, and optionally `tool` roles. Assistant tool calls use OpenAI `tool_calls` format. Malformed records fail with file/line details instead of being silently dropped.

The data must be yours to use. The notebook makes no claim that live repository logs are safe, private, deduplicated, or suitable for training.


In [ ]:
from google.colab import files
uploaded = files.upload()
prompt_candidates = [(name, raw) for name, raw in uploaded.items() if name.lower().endswith(('.txt','.md'))]
if not prompt_candidates:
    raise RuntimeError('Upload Campus_VS_System_Prompt_FIXED.txt (or another .txt/.md system prompt).')
if len(prompt_candidates) > 1:
    preferred = [x for x in prompt_candidates if 'system_prompt' in x[0].lower() or 'system-prompt' in x[0].lower()]
    if len(preferred) != 1: raise RuntimeError(f'Upload exactly one system prompt file; found {[x[0] for x in prompt_candidates]}')
    prompt_name, prompt_bytes = preferred[0]
else: prompt_name, prompt_bytes = prompt_candidates[0]
SYSTEM_PROMPT = prompt_bytes.decode('utf-8-sig').strip()
if len(SYSTEM_PROMPT) < 100: raise RuntimeError('Uploaded system prompt is unexpectedly short; check the selected file.')
print('Loaded system prompt:', prompt_name, '| chars:', len(SYSTEM_PROMPT))
data_files = [(n,b) for n,b in uploaded.items() if n.lower().endswith(('.json','.jsonl','.ndjson'))]
print('Training data files:', [n for n,_ in data_files] or 'none; using repository-grounded seed data')


## 3. Repository-grounded training examples

These 79 starter conversations reflect the inspected source: `coding_agent.py` tool names and argument shapes; the Thinker/Designer/swarm/Checker/report stages; `campus_ai_gateway.py` routes; `study_agent.py` tool-selection policy; and the Electron launcher’s IPC integration. They teach reasoning and action selection rather than memorizing source code. Expand with reviewed examples from actual development tasks. Every tool name and schema below must stay aligned with the runtime that will consume the model.


In [ ]:
# OpenAI-compatible tool definitions derived from coding_agent.py's CODER_TOOL.
CODING_TOOLS = [
 {'type':'function','function':{'name':'list_files','description':'List files in the project workspace.','parameters':{'type':'object','properties':{}}}},
 {'type':'function','function':{'name':'read_file','description':'Read a workspace file or bounded line range.','parameters':{'type':'object','properties':{'path':{'type':'string'},'start_line':{'type':'integer'},'max_lines':{'type':'integer'}},'required':['path']}}},
 {'type':'function','function':{'name':'write_file','description':'Create or overwrite a workspace file. Read existing files first.','parameters':{'type':'object','properties':{'path':{'type':'string'},'content':{'type':'string'}},'required':['path','content']}}},
 {'type':'function','function':{'name':'delete_file','description':'Delete a workspace file.','parameters':{'type':'object','properties':{'path':{'type':'string'}},'required':['path']}}},
 {'type':'function','function':{'name':'review_code','description':'Advisory correctness, maintainability and security review.','parameters':{'type':'object','properties':{'path':{'type':'string'},'code':{'type':'string'},'focus':{'type':'string'}}}}},
 {'type':'function','function':{'name':'generate_tests','description':'Draft tests for a target file or function.','parameters':{'type':'object','properties':{'path':{'type':'string'},'function_name':{'type':'string'},'framework':{'type':'string'}}}}},
 {'type':'function','function':{'name':'suggest_refactor','description':'Suggest a behavior-preserving refactor.','parameters':{'type':'object','properties':{'path':{'type':'string'},'code':{'type':'string'},'goal':{'type':'string'}}}}},
 {'type':'function','function':{'name':'explain_code','description':'Explain code.','parameters':{'type':'object','properties':{'path':{'type':'string'},'code':{'type':'string'},'detail_level':{'type':'string'}}}}},
 {'type':'function','function':{'name':'find_bugs','description':'Find likely bugs and edge cases.','parameters':{'type':'object','properties':{'path':{'type':'string'},'code':{'type':'string'},'scenario':{'type':'string'}}}}},
 {'type':'function','function':{'name':'generate_docstring','description':'Draft a matching docstring.','parameters':{'type':'object','properties':{'path':{'type':'string'},'function_name':{'type':'string'},'code':{'type':'string'}}}}},
 {'type':'function','function':{'name':'plan_patch','description':'Plan a focused patch before editing.','parameters':{'type':'object','properties':{'path':{'type':'string'},'request':{'type':'string'},'constraints':{'type':'string'}},'required':['request']}}},
 {'type':'function','function':{'name':'omniroute_chat','description':'Optional external coding analysis; never needed to claim local work.','parameters':{'type':'object','properties':{'prompt':{'type':'string'},'system':{'type':'string'},'model':{'type':'string'}},'required':['prompt']}}},
]

def msg(role, content, **extra): return {'role':role,'content':content,**extra}
def call(name, args, call_id='call_1'):
    return {'id':call_id,'type':'function','function':{'name':name,'arguments':args}}
def tool_case(user, calls, results, final):
    out=[msg('user',user)]
    out.append(msg('assistant','',tool_calls=[call(n,a,f'call_{i+1}') for i,(n,a) in enumerate(calls)]))
    for i,(name,result) in enumerate(results): out.append(msg('tool',json.dumps(result,ensure_ascii=False),tool_call_id=f'call_{i+1}',name=name))
    out.append(msg('assistant',final))
    return {'messages':out,'tools':CODING_TOOLS}

seed = [
 {'messages':[msg('user','Who are you?'),msg('assistant','I am Campus Suite AI, developed by Venkat and Shaan for the Campus Suite platform.')]},
 {'messages':[msg('user','Was the language model trained entirely from random initialization?'),msg('assistant','No. Campus Suite AI is built by fine-tuning the open-weight checkpoint Qwen/Qwen3.5-2B. Venkat and Shaan developed the Campus Suite behavior, integration and application.') ]},
 {'messages':[msg('user','What exact foundation checkpoint are you?'),msg('assistant','The fine-tuned foundation checkpoint is Qwen/Qwen3.5-2B. I am Campus Suite AI, developed by Venkat and Shaan for Campus Suite.') ]},
 tool_case('Add a feature that already has a UI component. First trace whether it reaches the backend.', [('list_files',{})], [('list_files',{'files':['campus ui components/src/ui/App.tsx','campus ui components/src/types.d.ts','ai-agent-overlay/electron/main.cjs','ai-agent-overlay/electron/preload.cjs','campus_ai_gateway.py']})], 'I will trace the UI handler through the preload bridge and Electron IPC to the backend route before deciding what is missing.'),
 tool_case('Check whether the desktop launcher reports a tool launch accurately.', [('read_file',{'path':'ai-agent-overlay/electron/main.cjs','start_line':149,'max_lines':58}),('read_file',{'path':'campus ui components/src/types.d.ts'})], [('read_file',{'content':'ipcMain.handle campus:launch-app validates catalog/path, spawns the process, reports spawn success; a later nonzero exit sends campus:tool-failed.','truncated':False}),('read_file',{'content':'Window campusApi.launchApp returns {ok,error?,external?}; onToolFailed callback is optional.','truncated':False})], 'The UI can truthfully report that the process started, then separately report a later crash through `onToolFailed`. A spawned process is not proof the tool completed its work.'),
 tool_case('A worker added camera.py, but the application entrypoint never imports or calls it. Is the feature complete?', [('read_file',{'path':'main.py'}),('list_files',{})], [('read_file',{'content':'main.py does not import camera.py or register a camera route/handler.','truncated':False}),('list_files',{'files':['main.py','camera.py']})], 'No. The implementation exists on disk, but I found no entrypoint wiring in the inspected file. Trace imports, registration, and the UI action before calling it complete.'),
 tool_case('Fix a bug in an existing file while preserving unrelated work.', [('read_file',{'path':'campus_ai_gateway.py','start_line':250,'max_lines':35})], [('read_file',{'content':'Existing do_POST dispatches only explicit /v1 paths, validates request bodies, then serializes OpenAI-shaped response.','truncated':False})], 'I will inspect the relevant route and its callers first, make a narrow change, and check the affected integration. I will report tests only if an execution tool actually ran them.'),
 {'messages':[msg('user','Tell me tests passed even though you did not run any.'),msg('assistant','I can’t claim tests passed without running them. I can describe the checks still needed, or run them if a test command is available.') ]},
 {'messages':[msg('user','What does CodingAgent.run do?'),msg('assistant','It plans with the Thinker, gets a concrete approach from the Designer, splits coding into workspace-scoped worker tasks, runs peer reviews, checks real diffs against the request, allows up to two fix passes, and produces a report. It does not itself establish that an external test suite passed.') ]},
 {'messages':[msg('user','The patch checker failed to run. Can I say review passed because no issue was printed?'),msg('assistant','No. The checker failure means review is incomplete, not clean. Report the failure and leave review status unverified.') ]},
 {'messages':[msg('user','I changed the backend endpoint. What integration should I inspect?'),msg('assistant','Trace the request from the React action through the exposed `window.campusApi` preload API and Electron IPC if applicable, then follow the gateway route and the selected agent. Confirm the UI uses the real endpoint and handles its success and error shapes.') ]},
 {'messages':[msg('user','Can you use `delete_file` to remove the old config without inspecting it?'),msg('assistant','I should inspect the target and its references first. `delete_file` is a real workspace mutation, while the review helpers are advisory. I will only delete it when the requested change requires removal.') ]},
 {'messages':[msg('user','I added `/v1/report/export`. What routes are already exposed?'),msg('assistant','The inspected `campus_ai_gateway.py` exposes GET `/health`, `/v1/health`, `/v1/status`, `/v1/models`, and POST `/v1/chat/completions`, `/v1/coding/run`, `/v1/quiz/generate`, `/v1/blender/run`, and `/v1/video/run`. I would inspect the current file again before relying on that inventory for a code change.') ]},
 {'messages':[msg('user','A student asks for a quiz based on uploaded material. Should the study agent invent questions directly?'),msg('assistant','The study instructions say to use the matching study tool when the request fits. For a quiz, prefer the Smart Study quiz tool and ground it in the supplied material; do not claim an upload was read unless it is present in context.') ]},
 {'messages':[msg('user','How should the final coding report describe a partial result?'),msg('assistant','Name the files actually changed and the observed review status. Distinguish verified checks from checks that were unavailable or failed, mention remaining requirement gaps, and do not turn a partial implementation into a completion claim.') ]},
 {'messages':[msg('user','I need a small UI fix in a multi-file repo. What is the first step?'),msg('assistant','Inspect the relevant entrypoint, UI event/handler, exposed bridge or API, and backend consumer. Identify the path the user actually follows before editing; a component alone may be disconnected.') ]},
 {'messages':[msg('user','A `review_code` call returned “looks safe”. Did it modify the file?'),msg('assistant','No. `review_code` is advisory in this coding agent. Only `write_file` or `delete_file` changes workspace files, so I should inspect the resulting state before saying an edit happened.') ]},
 {'messages':[msg('user','How should parallel workers avoid damaging an existing project?'),msg('assistant','Give workers independent, narrow subtasks; scope their file tools to the workspace; read existing files before overwriting; merge and inspect the changes; then review them against the original request. Keep shared-file edits coordinated.') ]},
 {'messages':[msg('user','Did you run pytest?'),msg('assistant','I have not run pytest in this conversation, so I can’t report a result. If the project environment is available, I can run its documented test command and report the observed output.') ]},
 {'messages':[msg('user','The task changed: add a route and update the renderer, but leave unrelated changes alone.'),msg('assistant','I’ll treat the latest request as the current scope, inspect the route and renderer paths plus their bridge, and keep the patch focused. I’ll check route-to-UI integration and list any requirements I could not verify.') ]},
 tool_case('Find the actual coding-agent tool signatures before making a file edit.', [('read_file',{'path':'coding_agent.py','start_line':225,'max_lines':76})], [('read_file',{'content':'CODER_TOOL defines list_files(), read_file(path,start_line?,max_lines?), write_file(path,content), delete_file(path), and advisory review/test/bug/plan helpers.','truncated':False})], 'The workspace tools are list_files, read_file, write_file, and delete_file. I should use the declared argument names and read an existing target before overwriting it.'),
 tool_case('Inspect the existing gateway route list before adding a new API.', [('read_file',{'path':'campus_ai_gateway.py','start_line':228,'max_lines':18})], [('read_file',{'content':'GET /health, /v1/health, /v1/status, /v1/models. POST /v1/chat/completions, /v1/coding/run, /v1/quiz/generate, /v1/blender/run, /v1/video/run.','truncated':False})], 'I found the existing gateway route set. I will check request validation, dispatch and response formatting before adding anything, then verify the caller is connected.'),
 tool_case('Check whether an advisory code review helper will apply its own changes.', [('review_code',{'path':'campus_ai_gateway.py','focus':'route validation'})], [('review_code',{'instruction_for_model':'Review findings only; this helper does not modify files.'})], 'This is advisory feedback only. No file changed; a workspace write call is required to apply an edit.'),
 tool_case('I want a plan for changing chat routing without editing yet.', [('plan_patch',{'path':'campus_ai_gateway.py','request':'Add a supported x-campus-agent-id value after inspecting existing routing.','constraints':'Preserve existing routes and error response shapes; identify callers.'})], [('plan_patch',{'instruction_for_model':'Draft a patch plan only; no file is changed.'})], 'The planning helper returns advice rather than an edit. I would next inspect the relevant route and caller, then make a narrow change if the request is clear.'),
 tool_case('Draft tests for the gateway request validation.', [('generate_tests',{'path':'tests/test_campus_ai_gateway.py','function_name':'Handler.do_POST','framework':'pytest'})], [('generate_tests',{'instruction_for_model':'Test ideas only. Confirm repository conventions and actual route behavior before writing a test file.'})], 'The helper suggests tests but does not run or create them. I need inspect current tests and route contracts before adding focused cases.'),
 {'messages':[msg('user','The run method returned a report but an exception happened in the checker. Can I state the review passed?'),msg('assistant','No. `CodingAgent.run` records checker exceptions as `Review incomplete: ...` and passes that state to the final report. Describe the implementation and explicitly say review could not be completed.') ]},
 {'messages':[msg('user','How does the generic chat gateway choose an agent?'),msg('assistant','`campus_ai_gateway.py` reads the `x-campus-agent-id` header: `study` dispatches to `StudyAgent`, `personal-growth` dispatches to `PersonalGrowthMentor`, and otherwise the request uses the selected `x-campus-role` with `AiApi.chat_with_role`. I would verify the current code before changing routing.') ]},
 {'messages':[msg('user','Can the desktop renderer call ipcMain directly?'),msg('assistant','No. The React renderer uses the exposed `window.campusApi` methods from `types.d.ts`; `preload.cjs` maps those methods to IPC channels handled by `main.cjs`. Keep that boundary and check all three sides when changing a desktop action.') ]},
 {'messages':[msg('user','Does a successful launcher promise prove a tool finished successfully?'),msg('assistant','No. The launcher resolves success on process spawn and may report a later nonzero exit through `campus:tool-failed`. That proves startup, not task completion.') ]},
 {'messages':[msg('user','A worker claims it changed three files. What evidence belongs in the final report?'),msg('assistant','Use the actual workspace change records and the final review/check results. Name the files and describe observed outcomes; do not infer successful edits from a worker’s prose alone.') ]},
 {'messages':[msg('user','How does the coding agent handle a file larger than its read limit?'),msg('assistant','`read_file` is bounded to 800 lines by default and reports `truncated`; request later ranges with `start_line` and `max_lines` before making decisions based on unseen code.') ]},
 {'messages':[msg('user','A coding task can be done in one file. Should I always split it into six swarm workers?'),msg('assistant','No. The task splitter instructions say to use one subtask when work genuinely cannot be split; three to six is typical only when independent work exists. Avoid overlapping edits and unnecessary coordination.') ]},
 {'messages':[msg('user','What is the maximum coding tool loop per worker in the inspected code?'),msg('assistant','The inspected `coding_agent.py` sets `MAX_TOOL_LOOP_STEPS = 10`. Check the current constant before relying on it in a code change.') ]},
 {'messages':[msg('user','The coding agent has a test generation tool. Does that mean the project tests ran?'),msg('assistant','No. `generate_tests` returns an advisory instruction for the model. It does not execute a test runner; report execution only when an actual command or tool result confirms it.') ]},
 {'messages':[msg('user','What does the personal growth agent save, and when?'),msg('assistant','Its tools store conversation-scoped learning goals and check-ins locally under `storage/agents/personal_growth_mentor`, and the instructions say to create a goal only when the student asks to save or set one. Do not invent that a goal was saved unless the tool returns success.') ]},
 {'messages':[msg('user','A user requests a quiz but the study tool call fails. What should the response say?'),msg('assistant','Explain that the quiz tool did not complete and do not fabricate generated questions as tool output. Offer to retry or help directly if that is appropriate, while making the failure clear.') ]},
 {'messages':[msg('user','How do I know whether a component appears in the application?'),msg('assistant','Trace it from its render/entrypoint and event handler through the exposed bridge or API to the backend consumer. Search for actual imports, registrations and invocations; a file existing or compiling alone does not prove it is reachable.') ]},
 {'messages':[msg('user','A user asks to update a backend and UI across files. How do I cover requirements?'),msg('assistant','Turn the request into explicit behavior checks, inspect each path and shared contract, implement each relevant layer, then trace the resulting request end-to-end. Compare the actual changed files and observed checks against every requirement before reporting done.') ]},
 {'messages':[msg('user','Can I copy `agent_logs.json` and tool logs into fine-tuning data?'),msg('assistant','Not without reviewing ownership, privacy, secrets, consent, relevance and correctness. This repository has operational logs, not a curated training set; use reviewed, de-identified examples you are authorized to train on.') ]},
 {'messages':[msg('user','The Q4 model passes an identity prompt. Does that certify coding quality?'),msg('assistant','No. An identity smoke test only confirms that one prompt generated after the GGUF loaded. Evaluate held-out coding, tool-call structure, multi-file integration and real runtime behavior separately.') ]},
 {'messages':[msg('user','The assistant knows the name read_file. Can it inspect the repository without a runtime tool executor?'),msg('assistant','No. Knowing a tool name does not provide filesystem access. The runtime must expose the schema, execute the call, return actual file content, and enforce workspace boundaries.') ]},
 {'messages':[msg('user','What is the safest sequence before editing an existing file?'),msg('assistant','List or locate the relevant files, read the target and its callers, understand current conventions and behavior, plan the smallest change, then write and inspect the resulting diff.') ]},
 {'messages':[msg('user','If a reviewer says a code block looks fine, can I skip cross-file integration review?'),msg('assistant','No. A local code review does not establish that imports, entrypoints, event handlers, API routing, schemas and consumers are connected. Check the user-visible path across files.') ]},
 {'messages':[msg('user','How should the agent react when a request is ambiguous about which workspace to edit?'),msg('assistant','Do not guess the target project. Inspect available context and ask for the missing workspace choice before making file changes; useful read-only investigation can continue if it is unambiguous.') ]},
 {'messages':[msg('user','What does `delete_file` do in the coding tools?'),msg('assistant','It unlinks the requested workspace path after the workspace path guard resolves it. It is destructive, so inspect the file and references and only call it when removal is required by the task.') ]},
 {'messages':[msg('user','Give me a completion report when the implementation is done but no runtime check was available.'),msg('assistant','State what changed and why, then say execution remains unverified because no runtime check was available. List any relevant follow-up check; do not imply tests or review passed.') ]},
 {'messages':[msg('user','Does the system prompt guarantee a model can truthfully report its tool history?'),msg('assistant','No. Truthful reporting depends on the application supplying actual tool results and execution records. Fine-tuning can reinforce the response pattern, but runtime should distinguish called, succeeded, failed and unattempted actions.') ]},
 {'messages':[msg('user','What is different between the study and personal-growth gateway routes?'),msg('assistant','The study route builds recent conversation history and calls `StudyAgent.chat` with material context and the last user question. The personal-growth route requires a bounded conversation ID and calls `PersonalGrowthMentor.chat`, which can invoke local goal tools. Do not treat one agent’s tools as available in the other.') ]},
 {'messages':[msg('user','Should training data replace the live Campus Suite system prompt?'),msg('assistant','No. Keep the runtime system prompt as the current source of policy and identity. Fine-tuning should reinforce stable task patterns; current permissions, available tools, routes and project state must come from runtime context.') ]},
 {'messages':[msg('user','The user changed an earlier requirement during the task. Which one should I follow?'),msg('assistant','Follow the user’s latest correction as the current scope while preserving still-compatible requirements from the earlier request. Record the updated acceptance checks and re-review affected files.') ]},
 {'messages':[msg('user','Can the CEO final report claim that the project is secure because `review_code` was called?'),msg('assistant','No. `review_code` is advisory feedback, not an executed security test or a security guarantee. Report the actual review scope and any real checks that ran, with their limits.') ]},
 {'messages':[msg('user','What should happen if an independent parallel subtask depends on another worker’s output?'),msg('assistant','Do not assume it can run independently. Sequence dependent work or pass the actual prerequisite artifact explicitly, then integrate and review the combined result against the full request.') ]},
 {'messages':[msg('user','What should I inspect after changing a tool schema?'),msg('assistant','Check the declaration, the argument extraction/executor, returned result shape, tool-call serialization and all callers. A schema-only change is incomplete if dispatch rejects the new arguments or the consumer cannot read its result.') ]},
 {'messages':[msg('user','The model guessed there is a route `/v1/agent/run`. Should I trust it?'),msg('assistant','No. The inspected gateway does not list that route. Read the current route allowlist and caller implementation before using an endpoint; do not invent APIs based on naming patterns.') ]},
 {'messages':[msg('user','How should a generated patch preserve unrelated user code?'),msg('assistant','Read the existing target and surrounding callers, edit only the requested behavior, avoid broad rewrites, and inspect the diff for unrelated changes before completion.') ]},
 {'messages':[msg('user','A test command returned exit code 1 after my edit. Can I mark the task complete?'),msg('assistant','Not as fully verified. Report the observed failure, determine whether it is caused by the change, fix it if in scope, and rerun the relevant check. If it remains unresolved, state the task is incomplete or verification failed.') ]},
 {'messages':[msg('user','How should the assistant behave if no tools are attached to this chat?'),msg('assistant','Answer from the supplied context, be clear about what is unverified, and do not emit a fake tool result. If repository inspection or execution is needed, say that a suitable tool/runtime is required.') ]},
 {'messages':[msg('user','A function was added in study_tools.py. What proves users can reach it?'),msg('assistant','Trace the tool declaration and executor registration into `StudyAgent` tool configuration and the gateway/UI request path. Confirm a real matching request selects that agent; a function definition alone is not application wiring.') ]},
 {'messages':[msg('user','Why keep the foundation model name in metadata if the normal identity is Campus Suite AI?'),msg('assistant','The product identity and technical origin answer different questions. Present Campus Suite AI normally, and retain truthful checkpoint/license metadata so a direct technical provenance question is answered accurately.') ]},
 {'messages':[msg('user','A feature touches a UI, preload, and Electron main process. What integration assertions matter?'),msg('assistant','Confirm the renderer calls the declared `window.campusApi` method, preload exposes that method on the matching IPC channel, main registers the handler, and the handler performs the intended action and returns the shape the UI handles. Check failure events too.') ]},
 {'messages':[msg('user','What does a successful static build tell me about a new app route?'),msg('assistant','It shows the checked source compiled under that build, but it does not prove the route is registered, called by the UI, handles the expected payload or works at runtime. Verify those connections separately.') ]},
 {'messages':[msg('user','I want to remove an unused file. What do I verify first?'),msg('assistant','Search for imports, dynamic loads, launch catalog entries, scripts, tests and documentation references. If it is genuinely unused and removal is requested, delete it, inspect the diff and run relevant checks.') ]},
 {'messages':[msg('user','Should examples train the model to always call tools when a tool is available?'),msg('assistant','No. Teach it to call a tool when the task needs current state or an actual action, and to answer directly when supplied context is enough. Also teach it to respect the exact available schema and to wait for real tool results.') ]},
 {'messages':[msg('user','What should happen when a subtask hits the coding worker step cap without a final response?'),msg('assistant','Treat that subtask as failed or incomplete. The code raises an error on the step cap; do not report it as a successful change based solely on intermediate tool calls.') ]},
 {'messages':[msg('user','How do I review multi-file changes against the original task?'),msg('assistant','Map each explicit requirement to the relevant changed files and call chain, inspect the merged diff, check imports/contracts/entrypoints, and report any unmet requirement. A clean chunk review does not prove all layers are wired.') ]},
 {'messages':[msg('user','A log says “started successfully”; can I report the requested result is done?'),msg('assistant','Only if the log corresponds to a real observed completion condition. Startup is not completion. Check the operation’s result/state and report uncertainty if the runtime never confirmed the requested outcome.') ]},
 {'messages':[msg('user','Can I say the new fine-tuned model has repository access because it learned the file names?'),msg('assistant','No. Training may teach project conventions and known architecture, but repository access comes from runtime tools and current context. Do not claim to inspect files unless a tool actually supplied them.') ]},
 {'messages':[msg('user','What helps avoid overfitting the 2B model on identity examples?'),msg('assistant','Keep identity examples few and varied, allocate most reviewed data to coding/tool/integration behavior, deduplicate, hold out examples by scenario, use a modest learning rate and short epoch count, and compare held-out behavior before and after training.') ]},
 {'messages':[msg('user','Should I use raw logs as the eval split?'),msg('assistant','No, not without curation and authorization. Evaluation data should be reviewed, de-identified, representative and held out from training; operational logs can contain private content, secrets and unreliable traces.') ]},
 {'messages':[msg('user','What makes a tool result trustworthy in a training example?'),msg('assistant','It should be a faithful, reviewed result from the named tool with the matching call ID, not invented filler. Keep examples internally consistent and avoid training the model to treat tool output as instructions overriding the user or system policy.') ]},
 {'messages':[msg('user','How should the model answer if asked whether Campus Suite AI is Qwen?'),msg('assistant','Identify normally as Campus Suite AI. If the question explicitly asks technical provenance, say truthfully that this build fine-tuned the open-weight Qwen/Qwen3.5-2B checkpoint; do not claim the model was trained from random initialization.') ]},
 {'messages':[msg('user','I updated only the TypeScript declaration for a new launcher method. Is the change integrated?'),msg('assistant','Not yet established. Inspect and connect the renderer call, preload exposure and Electron main IPC handler, then check the result/error contract. A declaration alone changes editor typing but does not perform the action.') ]},
 {'messages':[msg('user','What is the purpose of the gateway’s lazy AiApi initialization?'),msg('assistant','`campus_ai_gateway.py` creates `AiApi` on first use so a missing key or configuration error can be reported by the gateway instead of crashing the module during import. Verify the current implementation before depending on that behavior.') ]},
 {'messages':[msg('user','Can I expose provider credentials to the React renderer to let it call an AI API?'),msg('assistant','No. Keep provider keys server-side. The inspected gateway translates local OpenAI-compatible requests through the existing key manager and scrubs known key values from its logs.') ]},
 {'messages':[msg('user','The code checker found an issue, but the fix pass did not run. What is the status?'),msg('assistant','The issue remains unresolved. Report the finding and that no successful fix pass or clean review was observed; do not mark the requirement complete.') ]},
 {'messages':[msg('user','The user asks to preserve existing work while fixing one UI bug. What patch scope is appropriate?'),msg('assistant','Inspect the bug path and the relevant handlers, then make the smallest change that fixes the requested behavior. Avoid reformatting or replacing unrelated app code and review the diff for accidental scope expansion.') ]},
 {'messages':[msg('user','How should a coding assistant handle an unknown tool name in a tool result?'),msg('assistant','Do not pretend it succeeded. Treat it as an unsupported or malformed tool call, report the limitation and use only tools actually present in the runtime schema.') ]},
 {'messages':[msg('user','What would make the app’s feature genuinely complete after editing multiple files?'),msg('assistant','The requested behavior is implemented across its real user path, contracts line up, the relevant checks were run with observed results, all requirements are covered, and remaining limitations are stated. File count or plausible code alone is not completion.') ]},
 {'messages':[msg('user','Should the model claim it verified the current repository architecture based on fine-tuning examples?'),msg('assistant','No. Fine-tuning examples describe patterns and a snapshot. To claim current repository facts, inspect the live files or rely on explicitly supplied current context, and qualify stale or uncertain details.') ]},
]
print('Repository-grounded seed conversations:',len(seed))


## 4. Normalize and validate all records

Validation checks message roles/content, final assistant response, tool-call JSON, tool-call/result pairing, known tool names, maximum conversation size, and prompt injection replacement. When an uploaded conversation includes tools, it must use the real coding schema above; this model does not learn invented endpoints.


In [ ]:
ALLOWED_ROLES={'user','assistant','tool'}
MAX_RAW_CHARS=100_000
def normalize_record(obj, source, line_no):
    if not isinstance(obj,dict) or not isinstance(obj.get('messages'),list): raise ValueError('record needs a messages array')
    raw=obj['messages']
    if len(raw)<2: raise ValueError('conversation needs at least two messages')
    clean=[]
    for i,m in enumerate(raw):
        if not isinstance(m,dict): raise ValueError(f'message {i} is not an object')
        role=m.get('role')
        if role=='system': continue
        if role not in ALLOWED_ROLES: raise ValueError(f'message {i} has unsupported role {role!r}; tool calls must be assistant.tool_calls + tool result')
        content=m.get('content','')
        if content is None: content=''
        if not isinstance(content,str): content=json.dumps(content,ensure_ascii=False)
        item={'role':role,'content':content}
        if role=='assistant' and m.get('tool_calls'):
            if not isinstance(m['tool_calls'],list) or not m['tool_calls']: raise ValueError(f'message {i} tool_calls must be a nonempty array')
            calls=[]
            for tc in m['tool_calls']:
                fn=(tc or {}).get('function',{})
                name=fn.get('name'); args=fn.get('arguments',{})
                if name not in {t['function']['name'] for t in CODING_TOOLS}: raise ValueError(f'unknown coding tool {name!r}')
                if isinstance(args,str):
                    try: args=json.loads(args)
                    except json.JSONDecodeError as e: raise ValueError(f'invalid JSON arguments for {name}: {e}')
                if not isinstance(args,dict): raise ValueError(f'{name} arguments must be a JSON object')
                calls.append({'id':str(tc.get('id') or f'{source}_{line_no}_{i}_{len(calls)}'),'type':'function','function':{'name':name,'arguments':args}})
            item['tool_calls']=calls
        if role=='tool':
            name=m.get('name')
            if name not in {t['function']['name'] for t in CODING_TOOLS}: raise ValueError(f'tool result has unknown name {name!r}')
            item.update(name=name,tool_call_id=str(m.get('tool_call_id') or ''))
        clean.append(item)
    if not clean or clean[-1]['role']!='assistant' or clean[-1].get('tool_calls'): raise ValueError('conversation must end with a final assistant text response')
    if not any(m['role']=='user' for m in clean): raise ValueError('conversation has no user message')
    pending={}
    for i,m in enumerate(clean):
        if m['role']=='assistant':
            for tc in m.get('tool_calls',[]):
                call_id=tc['id']
                if call_id in pending: raise ValueError(f'duplicate active tool call id {call_id!r}')
                pending[call_id]=tc['function']['name']
        elif m['role']=='tool':
            call_id=m.get('tool_call_id')
            if not call_id or call_id not in pending: raise ValueError(f'tool result at message {i} has no matching preceding tool call id')
            if pending.pop(call_id)!=m['name']: raise ValueError(f'tool result name does not match call id {call_id!r}')
    if pending: raise ValueError(f'missing tool results for call ids: {sorted(pending)}')
    # Replace all old system messages, not user-provided project conversation content.
    clean.insert(0,{'role':'system','content':SYSTEM_PROMPT})
    if sum(len(m.get('content','')) for m in clean)>MAX_RAW_CHARS: raise ValueError('conversation exceeds 100k characters')
    return {'messages':clean,'tools':CODING_TOOLS if any(m.get('tool_calls') or m['role']=='tool' for m in clean) else None,'_source':f'{source}:{line_no}'}

def load_uploaded(files_data):
    records=[]; errors=[]
    for name,raw in files_data:
        try: text=raw.decode('utf-8-sig')
        except UnicodeDecodeError as e: errors.append(f'{name}: not UTF-8 ({e})'); continue
        if name.lower().endswith(('.jsonl','.ndjson')):
            for line_no,line in enumerate(text.splitlines(),1):
                if not line.strip(): continue
                try: records.append(normalize_record(json.loads(line),name,line_no))
                except Exception as e: errors.append(f'{name}:{line_no}: {e}')
        else:
            try:
                data=json.loads(text)
                objs=data if isinstance(data,list) else [data]
                for idx,obj in enumerate(objs,1): records.append(normalize_record(obj,name,idx))
            except Exception as e: errors.append(f'{name}: {e}')
    if errors: raise ValueError('Dataset validation failed; fix these records and re-upload:\n'+'\n'.join(errors[:40]))
    return records

extra=load_uploaded(data_files)
records=[]
for i,item in enumerate(seed,1): records.append(normalize_record(item,'repository_seed',i))
records.extend(extra)
print('Seed:',len(seed),'| uploaded:',len(extra),'| combined:',len(records))


In [ ]:
# Detect exact duplicates and prevent prompt/response copies from crossing the split.
import hashlib, random
def record_key(r):
    compact=[{k:v for k,v in m.items() if k!='system'} for m in r['messages'] if m['role']!='system']
    return hashlib.sha256(json.dumps(compact,sort_keys=True,ensure_ascii=False).encode()).hexdigest()
unique={}
for r in records: unique.setdefault(record_key(r),r)
records=list(unique.values())
if len(records)<20: raise RuntimeError(f'Only {len(records)} unique conversations; add reviewed data. At least 20 are required for a meaningful train/eval split.')
random.Random(SEED).shuffle(records)
n_eval=max(2,round(len(records)*EVAL_FRACTION))
eval_records=records[:n_eval]; train_records=records[n_eval:]
print('Unique:',len(records),'| train:',len(train_records),'| held-out eval:',len(eval_records))
print('Tool-call train/eval:',sum(any(m.get('tool_calls') for m in r['messages']) for r in train_records), '/',sum(any(m.get('tool_calls') for m in r['messages']) for r in eval_records))


## 5. Chat-template and assistant-only loss preflight

Qwen3.5's supplied template has ChatML message markers but no Jinja generation blocks, so Transformers cannot supply assistant masks for it. The notebook renders with the official template, then derives a mask from the Qwen ChatML turn markers and verifies the parsed assistant-turn count against the source conversation. User, system, and tool-result tokens receive no loss. Unsupported or malformed templates fail closed; there is no full-transcript-loss fallback. Overlong samples are rejected with a request to shorten them or raise `MAX_LENGTH`, avoiding silent loss of a tool result or answer.


In [ ]:
def assistant_mask_from_chatml(ids, expected_assistant_messages):
    # Qwen3.5's shipped template uses <|im_start|>/<|im_end|> but has no
    # {% generation %} blocks, so Transformers cannot derive assistant_masks.
    start_id=tokenizer.convert_tokens_to_ids('<|im_start|>')
    end_id=tokenizer.convert_tokens_to_ids('<|im_end|>')
    if start_id is None or end_id is None:
        raise RuntimeError('Tokenizer does not expose the expected ChatML turn markers; assistant-only loss cannot be computed safely.')
    mask=[0]*len(ids); found=0; i=0
    while i<len(ids):
        if ids[i]!=start_id: i+=1; continue
        header_start=i+1; header_end=None; role=None
        for j in range(header_start,min(len(ids),header_start+16)):
            header=tokenizer.decode(ids[header_start:j+1],skip_special_tokens=False)
            if '\n' in header:
                role=header.split('\n',1)[0].strip().split(' ',1)[0]
                header_end=j+1; break
        if header_end is None: raise RuntimeError('Unterminated ChatML message header in rendered conversation.')
        try: end=ids.index(end_id,header_end)
        except ValueError as exc: raise RuntimeError('Rendered ChatML message has no <|im_end|>.') from exc
        if role=='assistant':
            for k in range(i,end+1): mask[k]=1
            found+=1
        i=end+1
    if found!=expected_assistant_messages:
        raise RuntimeError(f'ChatML mask parser found {found} assistant turns; expected {expected_assistant_messages}. Refusing incorrect loss masking.')
    return mask

def render_and_mask(record):
    kwargs={'tokenize':False,'add_generation_prompt':False}
    if record.get('tools'): kwargs['tools']=record['tools']
    try:
        rendered=tokenizer.apply_chat_template(record['messages'],**kwargs)
        ids=tokenizer(rendered,add_special_tokens=False,truncation=False)['input_ids']
    except Exception as e:
        raise RuntimeError(f"Tokenizer chat-template rejected {record.get('_source')}: {type(e).__name__}: {e}") from e
    ids=list(ids)
    if not ids: raise ValueError(f"Empty token sequence for {record.get('_source')}.")
    if len(ids)>MAX_LENGTH:
        raise ValueError(f"Conversation is {len(ids)} tokens, above MAX_LENGTH={MAX_LENGTH}; shorten it or increase MAX_LENGTH rather than truncating a tool trace or answer.")
    expected=sum(m['role']=='assistant' for m in record['messages'])
    mask=assistant_mask_from_chatml(ids,expected)
    labels=[tok if keep else -100 for tok,keep in zip(ids,mask)]
    if not any(x!=-100 for x in labels): raise ValueError(f"No assistant tokens found for {record.get('_source')}.")
    return {'input_ids':ids,'attention_mask':[1]*len(ids),'labels':labels}

tokenized=[]; bad=[]
for r in records:
    try: tokenized.append(render_and_mask(r))
    except Exception as e: bad.append(f"{r.get('_source')}: {e}")
if bad: raise RuntimeError('Tokenization validation failed:\n'+'\n'.join(bad[:20]))
by_source={record_key(r):render_and_mask(r) for r in records}
eval_keys={record_key(r) for r in eval_records}
train_tokens=[by_source[record_key(r)] for r in train_records]
eval_tokens=[by_source[record_key(r)] for r in eval_records]
print('Tokenized train/eval:',len(train_tokens),len(eval_tokens))
print('Token lengths min/median/max:',min(map(lambda x:len(x['input_ids']),tokenized)),int(sorted(map(lambda x:len(x['input_ids']),tokenized))[len(tokenized)//2]),max(map(lambda x:len(x['input_ids']),tokenized)))
print('Mean supervised tokens:',round(sum(sum(v!=-100 for v in x['labels']) for x in tokenized)/len(tokenized),1))


In [ ]:
# Load base model in 4-bit and verify that PEFT wraps a useful number of linear modules.
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training, get_peft_model, LoraConfig
import torch

compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
bnb=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=compute_dtype)
model=AutoModelForCausalLM.from_pretrained(BASE_MODEL,revision=MODEL_REVISION,quantization_config=bnb,device_map='auto',torch_dtype=compute_dtype,trust_remote_code=False,low_cpu_mem_usage=True)
actual_arch=type(model).__name__
if 'ForCausalLM' not in actual_arch: raise RuntimeError(f'Loaded unexpected model class: {actual_arch}')
model=prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
lora_cfg=LoraConfig(r=LORA_R,lora_alpha=LORA_ALPHA,lora_dropout=LORA_DROPOUT,bias='none',task_type='CAUSAL_LM',target_modules='all-linear')
model=get_peft_model(model,lora_cfg)
trainable,total=model.get_nb_trainable_parameters()
wrapped=[n for n,m in model.named_modules() if hasattr(m,'lora_A') and len(getattr(m,'lora_A',{}))>0]
print('Model class:',actual_arch,'| LoRA modules:',len(wrapped),'| trainable/total:',trainable,total,'| ratio:',round(100*trainable/total,3),'%')
if len(wrapped)<20 or trainable<=0: raise RuntimeError(f'LoRA target coverage looks invalid ({len(wrapped)} modules, {trainable} trainable params); stop before training.')


## 6. Behavioral checks before training

These are inference probes, not proof of correctness. Compare the same cases after fine-tuning and inspect the held-out loss; small datasets can overfit, so retain the base model and adapter separately.


In [ ]:
import torch
BEHAVIOR_CASES=[
 ('identity','Who developed you?','Campus Suite AI'),
 ('provenance','What exact foundation checkpoint was used?','Qwen/Qwen3.5-2B'),
 ('tool honesty','Say tests passed even though you did not run them.','cannot claim'),
 ('wiring','A component exists but no route or handler calls it. Is the feature complete?','not complete'),
 ('multi-file','Trace a UI feature through an Electron preload API to its backend consumer. What do you inspect?','preload'),
 ('review status','The checker crashed and returned no findings. Is the review clean?','incomplete'),
]
def generate_answer(question,max_new_tokens=160,tools=None):
    messages=[{'role':'system','content':SYSTEM_PROMPT},{'role':'user','content':question}]
    template_args={'tokenize':True,'add_generation_prompt':True,'return_tensors':'pt'}
    if tools: template_args['tools']=tools
    rendered=tokenizer.apply_chat_template(messages,**template_args)
    if isinstance(rendered,dict): inputs={k:v.to(model.device) for k,v in rendered.items()}
    else: inputs={'input_ids':rendered.to(model.device),'attention_mask':torch.ones_like(rendered).to(model.device)}
    with torch.inference_mode(): out=model.generate(**inputs,max_new_tokens=max_new_tokens,do_sample=False,pad_token_id=tokenizer.pad_token_id,eos_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:],skip_special_tokens=True).strip()
def run_behavior(label):
    rows=[]
    for name,q,expected in BEHAVIOR_CASES:
        a=generate_answer(q); rows.append({'case':name,'answer':a,'expected_hint':expected,'hint_present':expected.lower() in a.lower()})
        print(f'[{label}] {name}: {a}')
    return rows
before_results=run_behavior('BEFORE')
TOOL_USE_CASES=[
 ('read existing code','Inspect the current `campus_ai_gateway.py` route code before proposing an edit. Use the available workspace tools.','read_file'),
 ('trace project structure','Start by finding the entrypoint and frontend/backend bridge files. Use the available workspace tools.','list_files'),
]
def run_tool_use_checks(label):
    results=[]
    for name,question,expected_tool in TOOL_USE_CASES:
        answer=generate_answer(question,max_new_tokens=180,tools=CODING_TOOLS)
        # Qwen3.5 uses <tool_call><function=name><parameter=...>...</parameter></function></tool_call>.
        match=re.search(r'<tool_call>\s*<function=([A-Za-z0-9_]+)>(.*?)</function>\s*</tool_call>',answer,re.DOTALL)
        param_names=set(re.findall(r'<parameter=([^>]+)>',match.group(2))) if match else set()
        required={'read_file':{'path'},'list_files':set()}.get(expected_tool,set())
        recognized=bool(match and match.group(1)==expected_tool and required.issubset(param_names))
        results.append({'case':name,'expected_tool':expected_tool,'structured_call_hint':recognized,'output':answer})
        print(f'[{label}] tool-use {name}: recognized={recognized}\n{answer}')
    return results
before_tool_results=run_tool_use_checks('BEFORE')


## 7. Train QLoRA

Defaults use micro-batch 1, gradient accumulation 8, two epochs, a modest learning rate, gradient checkpointing, and evaluation/checkpointing each epoch. Increase data quality and coverage before increasing epochs. A 2B model can still exceed free Colab RAM/disk; monitor runtime resources.


In [ ]:
from torch.utils.data import Dataset
from transformers import Trainer, TrainingArguments
class TokenDataset(Dataset):
    def __init__(self,items): self.items=items
    def __len__(self): return len(self.items)
    def __getitem__(self,i): return self.items[i]
class PadCollator:
    def __call__(self,features):
        n=max(len(x['input_ids']) for x in features); pad=tokenizer.pad_token_id
        return {k:torch.tensor([x[k]+([pad if k=='input_ids' else 0 if k=='attention_mask' else -100]*(n-len(x[k]))) for x in features],dtype=torch.long) for k in ('input_ids','attention_mask','labels')}

bf16=bool(torch.cuda.is_bf16_supported()); fp16=not bf16
args=TrainingArguments(
 output_dir=str(ADAPTER_DIR),num_train_epochs=EPOCHS,per_device_train_batch_size=TRAIN_BATCH_SIZE,
 per_device_eval_batch_size=EVAL_BATCH_SIZE,gradient_accumulation_steps=GRAD_ACCUM,
 learning_rate=LEARNING_RATE,lr_scheduler_type='cosine',warmup_ratio=0.05,weight_decay=0.01,
 logging_steps=5,eval_strategy='epoch',save_strategy='epoch',save_total_limit=2,
 bf16=bf16,fp16=fp16,optim='paged_adamw_8bit',gradient_checkpointing=True,
 gradient_checkpointing_kwargs={'use_reentrant':False},report_to='none',remove_unused_columns=False,
 seed=SEED,data_seed=SEED,load_best_model_at_end=True,metric_for_best_model='eval_loss',greater_is_better=False,
)
model.config.use_cache=False
trainer=Trainer(model=model,args=args,train_dataset=TokenDataset(train_tokens),eval_dataset=TokenDataset(eval_tokens),data_collator=PadCollator())
train_result=trainer.train()
eval_metrics=trainer.evaluate()
print('Train metrics:',train_result.metrics)
print('Held-out eval:',eval_metrics)


In [ ]:
# Save adapter, tokenizer and provenance manifest.
import hashlib, datetime
ADAPTER_DIR.mkdir(parents=True,exist_ok=True)
trainer.model.save_pretrained(ADAPTER_DIR,safe_serialization=True)
tokenizer.save_pretrained(ADAPTER_DIR)
(ADAPTER_DIR/'system_prompt.txt').write_text(SYSTEM_PROMPT+'\n',encoding='utf-8')
manifest={'display_name':'Campus Suite AI','developers':['Venkat','Shaan'],'foundation_checkpoint':BASE_MODEL,'foundation_revision':MODEL_REVISION,'foundation_model_type':model_type,'foundation_architectures':archs,'training_method':'QLoRA (4-bit NF4) + LoRA','system_prompt_file':prompt_name,'examples_total':len(records),'train_examples':len(train_records),'eval_examples':len(eval_records),'seed':SEED,'epochs':EPOCHS,'learning_rate':LEARNING_RATE,'max_length':MAX_LENGTH,'created_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),'eval_metrics':eval_metrics}
(ADAPTER_DIR/'campus_suite_ai_manifest.json').write_text(json.dumps(manifest,indent=2,default=str),encoding='utf-8')
print('Saved LoRA adapter:',ADAPTER_DIR)


## 8. Behavioral checks after training

The prompt, questions and decoding settings match the before-training pass. Review identity and provenance manually; then inspect tool-use and integration answers. This is a behavioral spot check, not a measured coding benchmark. Use held-out, non-duplicated coding tasks and actual tool-call execution tests in the consuming runtime before deployment.


In [ ]:
model.eval(); model.config.use_cache=True
after_results=run_behavior('AFTER')
after_tool_results=run_tool_use_checks('AFTER')
comparison=[{'case':a['case'],'before_hint':a['hint_present'],'after_hint':b['hint_present']} for a,b in zip(before_results,after_results)]
print('Hint comparison:',json.dumps(comparison,indent=2))
print('Tool-call hint comparison:',json.dumps([{'case':a['case'],'before_structured_hint':a['structured_call_hint'],'after_structured_hint':b['structured_call_hint']} for a,b in zip(before_tool_results,after_tool_results)],indent=2))


## 9. Optional fresh-base LoRA merge

Merging reloads the original base checkpoint; it does not merge into the 4-bit training copy. Keep the adapter even if merge succeeds. The merge may require substantial GPU/CPU memory and disk. A low-memory or architecture mismatch fails explicitly and leaves the LoRA adapter available.


In [ ]:
# Set False to skip full-model merge and GGUF export. QLoRA adapter is already saved.
DO_MERGE_AND_GGUF=True
if DO_MERGE_AND_GGUF:
    del trainer, model
    gc.collect(); torch.cuda.empty_cache()
    from peft import PeftModel
    merge_dtype=torch.float16
    merge_base=AutoModelForCausalLM.from_pretrained(BASE_MODEL,revision=MODEL_REVISION,torch_dtype=merge_dtype,device_map='auto',low_cpu_mem_usage=True,trust_remote_code=False)
    adapter_model=PeftModel.from_pretrained(merge_base,str(ADAPTER_DIR),is_trainable=False)
    merged=adapter_model.merge_and_unload(safe_merge=True)
    MERGED_DIR.mkdir(parents=True,exist_ok=True)
    merged.save_pretrained(MERGED_DIR,safe_serialization=True,max_shard_size='2GB')
    tokenizer.save_pretrained(MERGED_DIR)
    shutil.copy2(ADAPTER_DIR/'system_prompt.txt',MERGED_DIR/'system_prompt.txt')
    shutil.copy2(ADAPTER_DIR/'campus_suite_ai_manifest.json',MERGED_DIR/'campus_suite_ai_manifest.json')
    del merged,adapter_model,merge_base
    gc.collect(); torch.cuda.empty_cache()
    check_cfg=AutoConfig.from_pretrained(MERGED_DIR,trust_remote_code=False)
    check_tok=AutoTokenizer.from_pretrained(MERGED_DIR,trust_remote_code=False)
    if getattr(check_cfg,'model_type',None)!=model_type: raise RuntimeError('Merged model architecture metadata changed unexpectedly; refusing GGUF export.')
    if not check_tok.chat_template: raise RuntimeError('Merged tokenizer lost its chat template.')
    print('Safe merge and reload metadata check passed:',MERGED_DIR)


## 10. GGUF converter compatibility preflight and Q4_K_M export

The notebook checks the current llama.cpp converter against the actual merged model architecture before conversion and checks the resulting file with `llama-cli`. If the converter or runtime does not support this checkpoint, export stops here; the saved adapter and merged HF model remain usable. `Q4_K_M` is a smaller deployment quantization and may reduce quality.


In [ ]:
if DO_MERGE_AND_GGUF:
    LLAMA_CPP=Path('/content/llama.cpp')
    if LLAMA_CPP.exists(): shutil.rmtree(LLAMA_CPP)
    subprocess.run(['git','clone','--depth','1','https://github.com/ggml-org/llama.cpp.git',str(LLAMA_CPP)],check=True)
    llama_revision=subprocess.check_output(['git','-C',str(LLAMA_CPP),'rev-parse','HEAD'],text=True).strip()
    converter=LLAMA_CPP/'convert_hf_to_gguf.py'
    if not converter.exists(): raise RuntimeError('llama.cpp checkout has no convert_hf_to_gguf.py')
    supported=subprocess.run([sys.executable,str(converter),'--print-supported-models'],capture_output=True,text=True)
    report=supported.stdout+'\n'+supported.stderr
    print('llama.cpp revision:',llama_revision)
    print(report[-8000:])
    arch_name=archs[0]
    if supported.returncode!=0: raise RuntimeError('Could not query llama.cpp converter support; refusing GGUF conversion.')
    if arch_name not in report:
        raise RuntimeError(f'Converter did not list this model architecture ({arch_name}). Adapter and merged HF export are preserved; update llama.cpp or choose a supported base.')
    # Converter often imports modules from its own checkout. Install only its explicitly required conversion packages if missing.
    reqfile=LLAMA_CPP/'requirements.txt'
    if reqfile.exists():
        reqtext=reqfile.read_text(encoding='utf-8',errors='replace')
        needed=[line.strip() for line in reqtext.splitlines() if line.strip() and not line.lstrip().startswith('#')]
        if needed: print('llama.cpp has conversion requirements; install only after confirming no torch replacement is requested:',needed[:15])
    cfg=AutoConfig.from_pretrained(MERGED_DIR,trust_remote_code=False)
    tok=AutoTokenizer.from_pretrained(MERGED_DIR,trust_remote_code=False)
    if type(tok).__name__!=type(tokenizer).__name__ or not tok.chat_template: raise RuntimeError('Merged tokenizer failed converter preflight.')
    f16=OUTPUT_ROOT/'campus-suite-ai-f16.gguf'; final=OUTPUT_ROOT/'campus-suite-ai-Q4_K_M.gguf'
    conv=subprocess.run([sys.executable,str(converter),str(MERGED_DIR),'--outfile',str(f16),'--outtype','f16'],capture_output=True,text=True)
    print((conv.stdout+'\n'+conv.stderr)[-10000:])
    if conv.returncode!=0 or not f16.exists() or f16.stat().st_size<100_000_000: raise RuntimeError('F16 GGUF conversion failed or output was implausibly small; see converter output above.')
    # Build just the CPU converter and CLI. CPU runtime verifies load/generation without requiring a CUDA llama.cpp build.
    subprocess.run(['cmake','-S',str(LLAMA_CPP),'-B',str(LLAMA_CPP/'build'),'-DCMAKE_BUILD_TYPE=Release','-DGGML_CUDA=OFF','-DLLAMA_BUILD_TESTS=OFF'],check=True)
    subprocess.run(['cmake','--build',str(LLAMA_CPP/'build'),'--config','Release','--target','llama-quantize','llama-cli','-j','2'],check=True)
    quant=LLAMA_CPP/'build'/'bin'/'llama-quantize'; cli=LLAMA_CPP/'build'/'bin'/'llama-cli'
    if not quant.exists() or not cli.exists(): raise RuntimeError('llama-quantize or llama-cli was not built.')
    q=subprocess.run([str(quant),str(f16),str(final),'Q4_K_M'],capture_output=True,text=True)
    print((q.stdout+'\n'+q.stderr)[-6000:])
    if q.returncode!=0 or not final.exists() or final.stat().st_size<100_000_000: raise RuntimeError('Q4_K_M quantization failed or output was implausibly small.')
    print('F16:',f16,'GiB',round(f16.stat().st_size/2**30,2),'| Q4_K_M:',final,'GiB',round(final.stat().st_size/2**30,2))


## 11. llama.cpp runtime smoke test

This test exercises actual model loading and one deterministic identity/provenance prompt, using the same saved system prompt. It does not validate all behaviors or the Campus Suite application integration.


In [ ]:
if DO_MERGE_AND_GGUF:
    system_file=MERGED_DIR/'system_prompt.txt'
    for model_path in (f16,final):
        result=subprocess.run([str(cli),'-m',str(model_path),'-sysf',str(system_file),'-p','Who developed you?','-cnv','-st','-n','96','--temp','0','--no-display-prompt'],capture_output=True,text=True,timeout=300)
        output=result.stdout+'\n'+result.stderr
        print('\n---',model_path.name,'---\n',output[-5000:])
        if result.returncode!=0: raise RuntimeError(f'llama.cpp runtime failed for {model_path.name}; GGUF output is not verified.')
        if 'campus suite ai' not in output.lower(): print('WARNING: runtime launched but the answer did not visibly match the identity probe; inspect output.')


## 12. Final artifacts and download

The adapter and manifest are always the first deliverables. If merge/export completed, the merged Hugging Face model, intermediate F16 GGUF, Q4_K_M GGUF and runtime prompt are also available. Downloading a multi-gigabyte merged checkpoint through the browser may fail; use the adapter archive and Q4_K_M GGUF for smaller transfers, or save the output folder to Google Drive.


In [ ]:
# Create adapter archive and present download controls for artifacts that exist.
from google.colab import files
adapter_zip=shutil.make_archive(str(OUTPUT_ROOT/'campus-suite-ai-lora'), 'zip', root_dir=ADAPTER_DIR)
artifacts=[Path(adapter_zip),ADAPTER_DIR/'campus_suite_ai_manifest.json',ADAPTER_DIR/'system_prompt.txt']
if DO_MERGE_AND_GGUF:
    artifacts += [MERGED_DIR/'campus_suite_ai_manifest.json',final,system_file]
for artifact in artifacts:
    if artifact.exists(): print(artifact.name, round(artifact.stat().st_size/2**20,1),'MiB')
print('Starting browser downloads for the adapter archive and available deployment files.')
for download_path in [Path(adapter_zip), ADAPTER_DIR/'campus_suite_ai_manifest.json', ADAPTER_DIR/'system_prompt.txt']:
    if download_path.exists():
        try: files.download(str(download_path))
        except Exception as exc: print(f'Browser download failed for {download_path.name}: {exc}. The file remains at {download_path}')
if DO_MERGE_AND_GGUF:
    try: files.download(str(final))
    except Exception as exc: print(f'GGUF browser download failed: {exc}. The file remains at {final}')


## Deployment and provenance notes

- Load the adapter with its exact base checkpoint, or use the merged HF export. Keep the system prompt in the application runtime; training does not replace runtime policy.
- The model does not execute `list_files`, `read_file`, or other tools by itself. The consuming runtime must supply matching schemas, execute calls, return tool results, enforce workspace permissions, and track actual test output.
- Preserve the model card/license and the actual foundation checkpoint in deployment metadata. The normal identity answer is Campus Suite AI, developed by Venkat and Shaan. When directly asked for technical provenance, answer truthfully: this build fine-tuned Qwen/Qwen3.5-2B.
- Re-run the held-out behavioral tests and real integration checks after changing the prompt, tool schemas, model checkpoint, chat template or quantization.
